## 📦 Importation des bibliothèques

Ce bloc de code importe toutes les bibliothèques nécessaires pour entraîner, évaluer et suivre un modèle d’apprentissage par renforcement appliqué aux données financières.

### 🧩 Bibliothèques principales
- **Optuna** : utilisé pour l’optimisation automatique d’hyperparamètres.
- **NumPy** : permet de gérer efficacement les tableaux numériques.
- **Pandas** : facilite la manipulation et l’analyse de données tabulaires.
- **yFinance** : permet de télécharger des données financières (actions, indices, etc.) directement depuis Yahoo Finance.

### 🤖 Apprentissage par renforcement (RL)

- **Stable-Baselines3** : bibliothèque de référence pour les algorithmes d’apprentissage par renforcement.
- **PPO (Proximal** Policy Optimization) : algorithme RL utilisé pour entraîner un agent à prendre des décisions optimales.
- **DummyVecEnv** : permet de vectoriser les environnements RL (utile pour l’entraînement parallèle).
- **evaluate_policy** : fonction pour mesurer la performance de l’agent entraîné.
- **Monitor** : enregistre les statistiques d’exécution de l’environnement (récompenses, durée des épisodes, etc.).

### 🎮 Environnements et mémoire
- **Gymnasium** : framework pour créer et gérer des environnements RL personnalisés.
- **spaces** : sert à définir les actions et observations possibles dans l’environnement.
- **gc (Garbage Collector)** : utilisé pour libérer la mémoire pendant les expériences d’entraînement longues.

### 📊 Suivi et journalisation
- **Weights & Biases (wandb)** : plateforme de suivi des expériences (permet d’enregistrer les métriques, graphiques, et hyperparamètres).
- **WandbCallback** : connecte automatiquement Stable-Baselines3 à WandB pour un suivi en temps réel.

### 🧠 Détection d’anomalies et callbacks personnalisés

- **IForest (Isolation Forest)** : algorithme de détection d’anomalies, utile pour repérer des comportements inhabituels dans les données.
- **BaseCallback** : permet de créer ses propres callbacks pendant l’entraînement (ex. arrêt anticipé, sauvegarde personnalisée, etc.).
- **PyTorch (torch)** : bibliothèque de calcul matriciel et deep learning, utilisée en interne par Stable-Baselines3.

In [ ]:
import optuna
import numpy as np
import pandas as pd
import yfinance as yf
from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import DummyVecEnv
from stable_baselines3.common.evaluation import evaluate_policy
from stable_baselines3.common.monitor import Monitor
import gymnasium as gym
from gymnasium import spaces
import gc
import optuna
import numpy as np
from stable_baselines3 import PPO
from wandb.integration.sb3 import WandbCallback
import wandb
from alibi_detect.od import IForest
from stable_baselines3.common.callbacks import BaseCallback
import torch

## 🏦 Création d’un environnement Forex personnalisé

Ce code définit une classe ForexEnv — un environnement d’apprentissage par renforcement basé sur les données du marché des changes (Forex).
Elle suit la structure standard de Gymnasium, utilisée pour entraîner un agent (ex. avec PPO).

### ⚙️ Initialisation de l’environnement

```python
class ForexEnv(gym.Env):
    def __init__(self, df):
        super(ForexEnv, self).__init__()
        self.df = df.reset_index(drop=True)
        self.max_steps = len(df) - 1
        self.current_step = 0
```

- df contient les données historiques des prix (ex. téléchargées via yfinance).
- max_steps : nombre maximal d’étapes avant la fin d’un épisode.
- current_step : position actuelle dans la série temporelle.

### 🎮 Espace d’action et d’observation
```python
# Action : 0 = hold, 1 = buy, 2 = sell
self.action_space = spaces.Discrete(3)

# Observation : prix actuel et variation du prix précédent
self.observation_space = spaces.Box(low=-np.inf, high=np.inf, shape=(2,), dtype=np.float32)
```

- Espace d’action (action_space) : l’agent peut choisir entre
    - 0 → Hold (ne rien faire)
    - 1 → Buy (ouvrir une position longue)
    - 2 → Sell (ouvrir une position courte)
- Espace d’observation (observation_space) : le modèle observe
    - le prix actuel,
    - et la variation (delta) du prix par rapport à l’étape précédente.

### 🧭 Réinitialisation de l’environnement
```python
def reset(self, **kwargs):
    self.current_step = 0
    self.position = 0
    self.entry_price = 0.0
    obs = self._next_obs()
    info = {}
    return obs, info
```

- Remet l’environnement à zéro avant chaque épisode d’entraînement.
- Renvoie la première observation (prix et delta).
- position revient à 0 → aucune position ouverte.

### 🔍 Observation suivante
```python
def _next_obs(self):
    price = float(self.df.loc[self.current_step, 'Close'])
    delta = float(self.df.loc[self.current_step, 'Close'] - self.df.loc[self.current_step-1, 'Close']) if self.current_step > 0 else 0.0
    obs = np.array([price, delta], dtype=np.float32)
    return obs
```
- Récupère le prix de clôture courant (Close) et calcule le delta avec le prix précédent.
- Retourne un vecteur d’observation [prix, variation].

### 🚀 Fonction step(): une étape de trading

```python
def step(self, action):
    prev_price = float(self.df.loc[self.current_step, 'Close'])
    self.current_step += 1
    terminated = self.current_step >= len(self.df) - 1
    price = float(self.df.loc[self.current_step, 'Close'])
    delta = price - prev_price
```
- À chaque étape :
    - on avance d’un jour (current_step += 1),
    - on calcule la variation de prix (delta),
    - on vérifie si l’épisode est terminé (terminated).

### 💰 Gestion des actions et récompenses

```python
if action == 1:
    self.position = 1
elif action == 2:
    self.position = -1
elif action == 3:
    self.position = 0

reward = delta * self.position * 1000  # amplifié pour stabilité
```

- L’agent peut :
    - acheter (1) → parie sur une hausse,
    - vendre (2) → parie sur une baisse,
    - fermer sa position (3) → revient à neutre.
- La récompense (reward) est proportionnelle à :
Reward=Variation du prix×Position×1000
>(le facteur 1000 sert à rendre les récompenses plus stables et significatives numériquement).

### 🔁 Sorties de step()
```python
obs = np.array([price, delta], dtype=np.float32)
info = {}
truncated = False
return obs, reward, terminated, truncated, info
```

Retourne :
- la nouvelle observation,
- la récompense,
- un indicateur de fin d’épisode (terminated),
- un indicateur de troncature (truncated = False ici),
- un dictionnaire info pour des métadonnées éventuelles.

### 🧩 En résumé

Cet environnement simule un agent de trading Forex :

- il observe le prix et sa variation,
- il agit (acheter, vendre ou attendre),
- il reçoit une récompense selon le profit ou la perte,
- et il apprend à maximiser ses gains au fil des épisodes.

In [ ]:
class ForexEnv(gym.Env):
    def __init__(self, df):
        super(ForexEnv, self).__init__()
        self.df = df.reset_index(drop=True)
        self.max_steps = len(df) - 1
        self.current_step = 0

        # Action : 0 = hold, 1 = buy, 2 = sell
        self.action_space = spaces.Discrete(3)

        # Observation : prix actuel et delta précédent
        self.observation_space = spaces.Box(low=-np.inf, high=np.inf, shape=(2,), dtype=np.float32)

        self.position = 0  # 0 = neutral, 1 = long, -1 = short
        self.entry_price = 0.0

    def reset(self, **kwargs):
        self.current_step = 0
        self.position = 0
        self.entry_price = 0.0
        obs = self._next_obs()
        info = {}
        return obs, info

    def _next_obs(self):
        price = float(self.df.loc[self.current_step, 'Close'])
        delta = float(self.df.loc[self.current_step, 'Close'] - self.df.loc[self.current_step-1, 'Close']) if self.current_step > 0 else 0.0
        obs = np.array([price, delta], dtype=np.float32)
        return obs
    
    def step(self, action):
        prev_price = float(self.df.loc[self.current_step, 'Close'])
        self.current_step += 1
        terminated = self.current_step >= len(self.df) - 1
        price = float(self.df.loc[self.current_step, 'Close'])
        delta = price - prev_price

        # Actions : 0 hold, 1 buy, 2 sell, 3 close
        if action == 1:
            self.position = 1
        elif action == 2:
            self.position = -1
        elif action == 3:
            self.position = 0

        reward = delta * self.position * 1000  # amplifié pour stabilité

        obs = np.array([price, delta], dtype=np.float32)
        info = {}
        truncated = False
        return obs, reward, terminated, truncated, info


## 💾 Téléchargement et préparation des données Forex

Ce code récupère les données historiques du taux de change EUR/USD (Euro contre Dollar américain) à partir de Yahoo Finance, puis les prépare pour l’entraînement de l’agent de trading.

### 📥 Téléchargement des données
```python
df = yf.download("EURUSD=X", start="2020-01-01", end="2023-01-01", interval="1d")
```

- Utilise la bibliothèque yfinance pour télécharger les prix journaliers (interval = "1d") du taux de change EUR/USD.
- Les colonnes typiques sont :
    - Open, High, Low, Close → prix d’ouverture, maximum, minimum et clôture,
    - Adj Close → prix ajusté,
    - Volume → volume échangé (souvent nul pour le Forex).
- La période choisie va du 1er janvier 2020 au 1er janvier 2023.

### 🧹 Nettoyage du DataFrame
```python
df = df.dropna().reset_index(drop=True)
```

- dropna() : supprime les lignes contenant des valeurs manquantes (NaN).
- reset_index(drop=True) : réinitialise l’index du DataFrame après le nettoyage, pour que les lignes soient numérotées de 0 à n-1.

### ✅ Résultat :
df contient un jeu de données propre et complet de prix journaliers EUR/USD prêt à être utilisé par l’environnement ForexEnv.

In [ ]:
df = yf.download("EURUSD=X", start="2020-01-01", end="2023-01-01", interval="1d")
df = df.dropna().reset_index(drop=True)

## ⚙️ Création et optimisation de l’environnement PPO

Ce bloc définit deux éléments principaux :

- une fonction pour créer l’environnement Forex correctement initialisé,
- une fonction pour optimiser les hyperparamètres du modèle PPO à l’aide d’Optuna.

### 🧩 Fonction `make_env(seed=None)`

#### 🧠 Objectif:

Créer une fonction génératrice d’environnement Gym compatible avec Stable-Baselines3.

#### 🔍 Détails

- ForexEnv(df) : crée une nouvelle instance de l’environnement de trading défini précédemment.
- env.reset(seed=seed) : initialise l’environnement avec une graine aléatoire pour rendre les expériences 

**reproductibles.**

- Monitor(env) : enregistre les statistiques d’entraînement (récompenses, durée, etc.).
- La fonction retourne une fonction interne (_init) — une exigence de DummyVecEnv, qui attend un constructeur différé d’environnement.

### 🔧 Fonction `optimize_ppo(trial)`

#### 🧠 Objectif

Cette fonction est appelée par Optuna à chaque essai (trial) pour tester une combinaison différente d’hyperparamètres du modèle PPO.

### 🎲 Initialisation du seed
```python
random.seed(seed)
_np.random.seed(seed)
_torch.manual_seed(seed)
```

- Garantit la reproductibilité complète : chaque essai démarre avec la même graine aléatoire pour random, NumPy et PyTorch.
- La graine dépend du numéro du trial (trial.number), pour éviter toute corrélation entre essais.

### 🔍 Définition de l’espace d’hyperparamètres
```python
learning_rate = trial.suggest_loguniform("learning_rate", 1e-5, 5e-4)
batch_size = trial.suggest_categorical("batch_size", [64, 128, 256])
n_steps = max(batch_size, trial.suggest_categorical("n_steps", [1024, 2048, 4096]))
gamma = trial.suggest_uniform("gamma", 0.90, 0.999)
ent_coef = trial.suggest_uniform("ent_coef", 0.0, 0.02)
```

- Optuna propose automatiquement différentes valeurs pour :
    - learning_rate : taux d’apprentissage du modèle,
    - batch_size : taille des lots d’entraînement,
    - n_steps : nombre d’étapes avant chaque mise à jour,
    - gamma : facteur de discount pour les récompenses futures,
    - ent_coef : coefficient d’entropie (encourage l’exploration).
- L’utilisation de suggest_loguniform et suggest_categorical permet à Optuna d’explorer efficacement l’espace de recherche.

### 🧠 Création des environnements
```python
train_env = DummyVecEnv([make_env(seed)])
eval_env = DummyVecEnv([make_env(seed + 9999)])
```

- Deux environnements sont créés :
    - train_env : pour l’entraînement de l’agent,
    - eval_env : pour l’évaluation indépendante.
- L’utilisation de graines différentes garantit des conditions d’évaluation non biaisées.

### 🏋️ Entraînement du modèle PPO
```python
model = PPO(
    "MlpPolicy",
    train_env,
    learning_rate=learning_rate,
    n_steps=n_steps,
    batch_size=batch_size,
    gamma=gamma,
    ent_coef=ent_coef,
    tensorboard_log="./logs/optuna_forex",
    verbose=0,
)
```
- PPO (Proximal Policy Optimization) : algorithme RL robuste et stable.
- MlpPolicy : réseau de neurones multi-couches utilisé pour apprendre la stratégie.
- tensorboard_log : permet de suivre l’entraînement via TensorBoard.
```python
model.learn(total_timesteps=100_000, callback=NaNStopCallback())
```

- L’agent s’entraîne pendant 100 000 étapes.
- Le callback NaNStopCallback (défini plus bas) interrompt l’entraînement si des poids deviennent NaN (instabilité numérique).

### 🧪 Évaluation du modèle
```python
mean_reward, _ = evaluate_policy(model, eval_env, n_eval_episodes=10, deterministic=False)
```
- Évalue la performance moyenne du modèle sur 10 épisodes.
- deterministic=False : permet une évaluation plus réaliste en introduisant de la variance dans les actions.

### 💾 Sauvegarde du modèle et nettoyage
```python
tmp_path = f"tmp_models/trial_{trial.number}.zip"
model.save(tmp_path)
trial.set_user_attr("model_path", tmp_path)
trial.set_user_attr("mean_reward", float(mean_reward))
```
- Sauvegarde le modèle entraîné pour ce trial (avec son score moyen).
- Enregistre ces informations dans les attributs utilisateur Optuna, pour analyse ultérieure.
```python
model.env.close()
eval_env.close()
del model, train_env, eval_env
import gc; gc.collect()
```

- Ferme proprement les environnements et libère la mémoire (important pour de nombreux essais).

### 🛑 Callback : `NaNStopCallback`
```python
class NaNStopCallback(BaseCallback):
    def _on_step(self):
        for param in self.model.policy.parameters():
            if torch.isnan(param).any():
                return False  # stop training
        return True
```

- Vérifie à chaque étape si des poids du modèle deviennent NaN.
- Si c’est le cas, l’entraînement est arrêté immédiatement pour éviter la propagation d’erreurs.
- Permet d’assurer la stabilité numérique du processus d’optimisation.

### 📈 Suivi avec TensorBoard
Pour visualiser l’entraînement, ouvrir TensorBoard dans le terminal à partir du dossier de logs :
```bash
tensorboard --logdir ./logs/optuna_forex
```
Ensuite, ouvrir l’URL indiquée (souvent http://localhost:6006) dans un navigateur.

#### 🔑 Courbes importantes
1) **rollout/ep_rew_mean** : moyenne des récompenses par épisode
    - Indique si l’agent apprend à maximiser ses gains.
2) **rollout/ep_len_mean** : longueur moyenne des épisodes
    - Permet de vérifier si l’agent termine ses épisodes correctement ou s’arrête prématurément.
3) **policy/entropy_loss** : entropie de la politique
    - Une entropie trop faible → agent trop déterministe → risque de surapprentissage.
4) **policy/clip_fraction** : proportion des mises à jour affectées par le clip PPO
    - Permet de suivre si l’agent explore correctement ou reste bloqué dans certaines actions.

>Interprétation :
>- Une récompense moyenne qui augmente régulièrement est le signe que l’agent apprend.
>- Une entropie trop faible peut nécessiter d’augmenter ent_coef pour encourager l’exploration.

### ✅ En résumé

Ce bloc met en place :

- un environnement Forex simulé,
- un processus d’optimisation automatique des hyperparamètres du PPO,
- une séparation propre entre entraînement et évaluation,
- et un mécanisme de sécurité contre les instabilités numériques.

In [ ]:
def make_env(seed=None):
    def _init():
        env = ForexEnv(df)
        # Gymnasium reset peut accepter seed ; sinon on applique seeds globalement
        try:
            env.reset(seed=seed)
        except TypeError:
            pass
        return Monitor(env)
    return _init

# === OPTIMISATION DES HYPERPARAMÈTRES ===
def optimize_ppo(trial):
    import random, numpy as _np, torch as _torch
    seed = 1000 + trial.number
    random.seed(seed)
    _np.random.seed(seed)
    _torch.manual_seed(seed)

    learning_rate = trial.suggest_loguniform("learning_rate", 1e-5, 5e-4)
    batch_size = trial.suggest_categorical("batch_size", [64, 128, 256])
    n_steps = max(batch_size, trial.suggest_categorical("n_steps", [1024, 2048, 4096]))
    gamma = trial.suggest_uniform("gamma", 0.90, 0.999)
    ent_coef = trial.suggest_uniform("ent_coef", 0.0, 0.02)

    # env d'entraînement (seedé) et env d'évaluation séparé
    train_env = DummyVecEnv([make_env(seed)])
    eval_env = DummyVecEnv([make_env(seed + 9999)])

    model = PPO(
        "MlpPolicy",
        train_env,
        learning_rate=learning_rate,
        n_steps=n_steps,
        batch_size=batch_size,
        gamma=gamma,
        ent_coef=ent_coef,
        tensorboard_log="./logs/optuna_forex",
        verbose=0,
    )

    model.learn(total_timesteps=100_000, callback=NaNStopCallback())

    # evaluation sur env séparé, plus d'épisodes et non-deterministe pour plus de variance informative
    mean_reward, _ = evaluate_policy(model, eval_env, n_eval_episodes=10, deterministic=False)

    tmp_path = f"tmp_models/trial_{trial.number}.zip"
    model.save(tmp_path)
    trial.set_user_attr("model_path", tmp_path)
    trial.set_user_attr("mean_reward", float(mean_reward))

    model.env.close()
    eval_env.close()
    del model, train_env, eval_env
    import gc; gc.collect()

    return float(mean_reward)

class NaNStopCallback(BaseCallback):
    def _on_step(self):
        for param in self.model.policy.parameters():
            if torch.isnan(param).any():
                return False  # stop training
        return True

## 🧠 Lancement de l’optimisation Optuna et sélection du meilleur modèle

Ce bloc de code lance l’étude Optuna pour optimiser automatiquement les hyperparamètres du modèle PPO, puis charge ou réentraîne le meilleur modèle obtenu.

### 🚀 Création et exécution de l’étude Optuna
```python
study = optuna.create_study(direction="maximize")
study.optimize(optimize_ppo, n_trials=20)
```
- `optuna.create_study(direction="maximize")` : crée une nouvelle étude qui cherche à maximiser la récompense moyenne (objectif du RL).

- `study.optimize(...)` : lance le processus d’optimisation.
    - Appelle la fonction optimize_ppo() (définie précédemment)
    - pendant 20 essais (n_trials=20),
    - chacun testant une combinaison différente d’hyperparamètres.

👉 Au total, Optuna entraîne 20 modèles PPO différents, puis garde le plus performant.

### 🏆 Sélection du meilleur essai
```python
best_trial = max(
    study.get_trials(deepcopy=False),
    key=lambda t: t.value if t.value is not None else float('-inf')
)
```

- Récupère tous les essais (get_trials) et sélectionne celui avec la meilleure valeur de récompense moyenne (t.value).
- L’argument deepcopy=False évite de dupliquer inutilement les objets en mémoire.
- Si une valeur est manquante (None), elle est remplacée par -inf pour ne pas casser le tri.

### 💾 Chargement du meilleur modèle
```python
best_model_path = best_trial.user_attrs.get("model_path")
from stable_baselines3 import PPO
if best_model_path:
    best_model = PPO.load(best_model_path)
```

- Récupère le chemin du modèle sauvegardé lors de l’optimisation.
- Si le fichier existe, il est directement chargé depuis le disque grâce à PPO.load().

### 🔁 Réentraînement si nécessaire
```python
else:
    best_params = best_trial.params
    env = DummyVecEnv([lambda: Monitor(ForexEnv(df))])
    best_model = PPO("MlpPolicy", env, **best_params)
    best_model.learn(total_timesteps=500_000)
```

- Si aucun modèle sauvegardé n’est trouvé :
    - Récupère les meilleurs hyperparamètres (best_trial.params),
    - Crée un nouvel environnement Forex,
    - Réentraîne un modèle PPO avec ces paramètres pendant 500 000 étapes.
- Cela garantit d’avoir un modèle final même si la sauvegarde a échoué.

### 📊 Affichage des résultats
```python
print("✅ Best trial:")
print("  Reward:", best_trial.value)
print("  Params:", best_trial.params)
```
- Affiche la récompense moyenne du meilleur essai (Reward)
- et les hyperparamètres correspondants (Params).

### ✅ En résumé

Ce bloc :

- lance l’optimisation automatique des hyperparamètres du modèle PPO,
- sélectionne le meilleur essai,
- charge ou réentraîne le modèle optimal,
- et affiche les performances obtenues.

En d’autres termes, c’est la boucle complète d’expérimentation du projet :
>Optuna explore → PPO apprend → le meilleur agent Forex est conservé 🎯

In [ ]:
study = optuna.create_study(direction="maximize")
study.optimize(optimize_ppo, n_trials=20)

best_trial = max(study.get_trials(deepcopy=False), key=lambda t: t.value if t.value is not None else float('-inf'))
best_model_path = best_trial.user_attrs.get("model_path")
from stable_baselines3 import PPO
if best_model_path:
    best_model = PPO.load(best_model_path)
else:
    # re-entraîner à partir des best_params
    best_params = best_trial.params
    env = DummyVecEnv([lambda: Monitor(ForexEnv(df))])
    best_model = PPO("MlpPolicy", env, **best_params)
    best_model.learn(total_timesteps=500_000)
print("✅ Best trial:")
print("  Reward:", best_trial.value)
print("  Params:", best_trial.params)

## 🎯 Entraînement final du modèle PPO avec suivi W&B

Ce bloc lance l’entraînement final du meilleur modèle PPO en utilisant les meilleurs hyperparamètres trouvés par Optuna, tout en intégrant la plateforme de suivi Weights & Biases (W&B).

### 🧩 Préparation de l’environnement
```python
best_params = best_trial.params
env = DummyVecEnv([make_env()])
```

- Récupère les meilleurs hyperparamètres de l’étude Optuna (best_trial.params).
- Crée un environnement vectorisé (DummyVecEnv) pour l’entraînement.
- Remarque : on passe la fonction make_env(), pas son appel direct, pour que Stable-Baselines3 puisse initialiser l’environnement correctement.

### 📊 Initialisation de W&B
```python
wandb.init(
    mode="offline", 
    project="forex-ppo-hallucination",
    config=best_params,
)
```
- Initialise un run Weights & Biases pour suivre l’entraînement.
- mode="offline" : les données sont enregistrées localement, mais la structure W&B est maintenue.
- project="forex-ppo-hallucination" : nom du projet pour organiser les runs.
- config=best_params : enregistre automatiquement les hyperparamètres utilisés.

### 🏋️ Entraînement du modèle
```python
best_model = PPO("MlpPolicy", env, **best_params, verbose=0)
best_model.learn(
    total_timesteps=500_000,
    callback=WandbCallback(
        gradient_save_freq=100,
        model_save_path=f"models/{wandb.run.name}",
        verbose=2,
    ),
)
```
- PPO("MlpPolicy", env, **best_params) : crée un modèle PPO avec le réseau multi-couches (MlpPolicy) et les hyperparamètres optimaux.
- total_timesteps=500_000 : entraîne l’agent sur 500 000 étapes.
- WandbCallback :
    - enregistre les gradients tous les 100 pas,
    - sauvegarde le modèle dans un dossier lié au nom du run,
    - affiche les logs détaillés pendant l’entraînement.

### 💾 Sauvegarde finale et fermeture de W&B
```python
best_model.save("models/best_ppo_forex.zip")
wandb.finish()
```

- Sauvegarde le modèle final entraîné sous models/best_ppo_forex.zip.
- Termine proprement la session W&B, en enregistrant toutes les métriques et artefacts.

### ✅ En résumé

Ce bloc permet de :

1) Initialiser le modèle avec les meilleurs paramètres Optuna,
2) Entraîner l’agent PPO sur l’environnement Forex simulé,
3) Suivre l’entraînement avec W&B (métriques, gradients, checkpoints),
4) Sauvegarder le modèle final prêt à être utilisé pour l’évaluation ou le trading simulé.

>🔹 À ce stade, l’agent est prêt pour une évaluation sur des données unseen ou pour être utilisé dans une stratégie de trading simulée.

In [ ]:
# === ENTRAÎNEMENT FINAL AVEC W&B ===
best_params = best_trial.params
# Correction ici : on passe la fonction make_env() et non make_env
env = DummyVecEnv([make_env()])  

wandb.init(
    mode="offline", 
    project="forex-ppo-hallucination",
    config=best_params,
)

best_model = PPO("MlpPolicy", env, **best_params, verbose=0)
best_model.learn(
    total_timesteps=500_000,
    callback=WandbCallback(
        gradient_save_freq=100,
        model_save_path=f"models/{wandb.run.name}",
        verbose=2,
    ),
)
best_model.save("models/best_ppo_forex.zip")
wandb.finish()

## 🕵️ Détection des hallucinations dans la politique PPO

Cette fonction évalue si le modèle PPO entraîne des comportements incohérents ou "hallucinatoires" pendant la simulation sur l’environnement Forex.

Elle utilise l’algorithme Isolation Forest (IForest) pour détecter des actions anormales par rapport à la distribution normale de l’agent.

### ⚙️ Définition de la fonction
```python
def detect_hallucinations(model, env, n_steps=5000, threshold=0.05):
```

- model : le modèle PPO entraîné.
- env : environnement Forex simulé.
- n_steps : nombre de pas de simulation pour collecter des données d’actions.
- threshold : sensibilité pour la détection d’anomalies dans IForest.

### 🔄 Simulation et collecte des actions
```python
obs = env.reset()
actions = []
rewards = []

for _ in range(n_steps):
    action, _ = model.predict(obs, deterministic=False)
    obs, reward, done, info = env.step(action)
    actions.append(action.flatten())
    rewards.append(np.mean(reward))
    if done:
        obs = env.reset()
```

- L’agent joue n_steps étapes dans l’environnement :
    - predict(..., deterministic=False) : actions stochastiques pour explorer la politique réelle.
    - Les actions et récompenses sont collectées dans des listes.
    - Si l’épisode se termine (done), l’environnement est réinitialisé.

### 🧠 Détection d’anomalies avec IForest
```python
actions = np.array(actions)
od = IForest(threshold=threshold)
od.fit(actions)
preds = od.predict(actions)
```

- Convertit les actions en tableau NumPy.
- Crée un modèle Isolation Forest (IForest) pour identifier les actions atypiques.
- fit(actions) : apprend la distribution normale des actions.
- predict(actions) : marque chaque action comme normale ou anomalie.

### ⚠️ Calcul et affichage du ratio d’anomalies
```python
anomaly_ratio = preds["data"]["is_outlier"].mean()
print(f"⚠️ Anomaly ratio (hallucination risk): {anomaly_ratio:.2%}")
```

- anomaly_ratio : proportion d’actions considérées comme hors distribution.
- Permet d’évaluer le risque de comportement incohérent de la politique.

### ✅ Interprétation
```python
if anomaly_ratio > 0.1:
    print("🚨 Possible policy hallucination detected! (actions incohérentes)")
else:
    print("✅ Policy seems stable and consistent.")
```
- Si plus de 10 % des actions sont anormales → alerte sur hallucination.
- Sinon, la politique est jugée stable et cohérente.

### 🔙 Retour de la fonction
```python
return anomaly_ratio
```
- Retourne le ratio d’actions anormales pour un suivi quantitatif ou comparaison entre modèles.

### 🧩 En résumé

Cette fonction permet de :

1) Simuler l’agent sur un certain nombre de pas.
2) Collecter ses actions et récompenses.
3) Détecter des actions incohérentes via Isolation Forest.
4) Informer sur la stabilité de la politique et son risque d’hallucination.
>🔹 Utile pour éviter que l’agent PPO ne prenne des décisions irrationnelles ou erratiques sur des données réelles ou simulées.

In [ ]:
def detect_hallucinations(model, env, n_steps=5000, threshold=0.05):
    obs = env.reset()
    actions = []
    rewards = []

    for _ in range(n_steps):
        action, _ = model.predict(obs, deterministic=False)
        obs, reward, done, info = env.step(action)
        actions.append(action.flatten())
        rewards.append(np.mean(reward))
        if done:
            obs = env.reset()

    actions = np.array(actions)
    rewards = np.array(rewards)

    # === Détection d’anomalies sur les actions ===
    od = IForest(threshold=threshold)
    od.fit(actions)
    preds = od.predict(actions)

    anomaly_ratio = preds["data"]["is_outlier"].mean()
    print(f"⚠️ Anomaly ratio (hallucination risk): {anomaly_ratio:.2%}")

    if anomaly_ratio > 0.1:
        print("🚨 Possible policy hallucination detected! (actions incohérentes)")
    else:
        print("✅ Policy seems stable and consistent.")

    return anomaly_ratio

## 🧪 Test de détection des hallucinations sur le modèle final

Ce bloc met en pratique la fonction detect_hallucinations pour évaluer la stabilité du modèle PPO entraîné sur l’environnement Forex.

### 🔄 Création de l’environnement de test
```python
env_test = DummyVecEnv([lambda: Monitor(ForexEnv(df))])
```

- Crée un environnement vectorisé pour l’évaluation, identique à l’environnement d’entraînement mais séparé.
- Monitor permet d’enregistrer les statistiques lors de la simulation.
- lambda: ForexEnv(df) : assure que chaque appel crée une nouvelle instance propre.

### 🕵️ Détection des hallucinations
```python
detect_hallucinations(best_model, env_test, n_steps=5000, threshold=0.05)
```

- Simule le modèle best_model pendant 5000 pas.
- Détecte les actions incohérentes avec un seuil de 5 % pour l’Isolation Forest.
- Affiche :
    - le ratio d’anomalies dans les actions,
    - un message d’alerte si la politique semble instable.

### 💾 Fermeture de l’environnement
```python
env.close()
```

- Libère proprement les ressources de l’environnement, évitant fuites mémoire ou conflits lors de nouvelles simulations.

### ✅ En résumé

Ce bloc permet de :

1) Créer un environnement de test indépendant,
2) Évaluer la stabilité du modèle final PPO,
3) Identifier les risques d’hallucination (actions incohérentes ou erratiques),
4) Assurer la fiabilité du modèle avant déploiement ou analyse.
>🔹 C’est l’étape finale pour vérifier que l’agent agit de manière cohérente sur le marché simulé.

In [ ]:
env_test = DummyVecEnv([lambda: Monitor(ForexEnv(df))])

detect_hallucinations(best_model, env_test, n_steps=5000, threshold=0.05)
env.close()
